# 📌 Traccia: Regressione - Kernel Ridge vs Decision Tree con Manifold Learning
- Tipo di problema: Regressione
- Dataset: Dataset sintetico generato con make_swiss_roll da sklearn.datasets, trasformato in un problema di regressione, n_samples=1000, Si genera una variabile target y come funzione non lineare del parametro di roll (t) più un po' di rumore gaussiano

1. Pipeline 1:
- Preprocessing: StandardScaler
- Riduzione dimensionalità: Isomap (n_components=1)
- Modello: KernelRidge (kernel RBF, ottimizzazione di alpha e gamma)

2. Pipeline 2:
- Preprocessing: MinMaxScaler
- Riduzione dimensionalità: LocallyLinearEmbedding (n_components=2)
- Modello: DecisionTreeRegressor (ottimizzazione di max_depth)

- Metrica di valutazione:
R² (score di determinazione)

- Valutazione tramite Nested Cross-Validation:
    - Outer CV: 5-fold
    - Inner CV: 3-fold per tuning dei parametri

✅ Obiettivo dello studente:
- Generare il dataset swiss_roll e costruire la variabile target y
- Implementare le due pipeline complete con preprocessing, manifold learning, regressore
- Applicare nested CV per selezione dei parametri e valutazione in R²

Discutere i risultati in termini di capacità dei modelli di catturare la non linearità implicita nei dati

📎 Nota didattica:
Questa traccia:
- Introduce una situazione concreta dove il manifold learning è utile, in quanto i dati giacciono su una varietà bassa dimensionalità
- Confronta un modello kernel-based regolarizzato (Kernel Ridge) con uno più interpretabile e locale (Decision Tree)
- Mette in risalto come la riduzione non lineare della dimensionalità possa influenzare le prestazioni in regressione
- È fattibile in 20 minuti grazie all’uso di dataset sintetico semplice ma non banale

In [11]:
import import_ipynb
from utilities.functions import nested_cv, best_manifold, plot_embedding, train_final_model_from_nested_cv

# Generazione del dataset

In [2]:
from sklearn.datasets import make_swiss_roll
import numpy as np

# swiss roll con n_samples = 1000 e variabile target y generata come funzione non lineare del parametro roll (t) più un po' di rumore gaussiano
X, t = make_swiss_roll(n_samples = 1000, noise = 0.1)

y = np.sin(t) + np.random.normal(0, 0.1, t.shape)

print("X shape", X.shape)
print("y shape", y.shape)

X shape (1000, 3)
y shape (1000,)


# Dataset splitting

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

## Pipeline 1
   - Preprocessing: StandardScaler
   - Riduzione dimensionalità: Isomap (n_components=1)
   - Modello: KernelRidge (kernel RBF, ottimizzazione di alpha e gamma)

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import Isomap
from sklearn.kernel_ridge import KernelRidge

kr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('isomap', Isomap()),
    ('kernel_ridge', KernelRidge())
])

kr_params_grid = {
    'isomap__n_neighbors': [3, 5, 10],
    'isomap__n_components': [2, 3],
    'kernel_ridge__alpha': [0.01, 0.1, 1, 10],
    'kernel_ridge__gamma': [0.01, 0.1, 1, 10]
}

In [5]:
kr_result_nested_cv = nested_cv(
    model = kr_pipeline,
    param_grid = kr_params_grid,
    X_train = X_train,
    y_train = y_train,
    outer_splits = 5,
    inner_splits = 3,
    scoring = ['r2'])

--- Start time: 0.00 seconds ---

Performing Outer Fold 1/5
Performing GridSearchCV (optimizing for 'r2')...
  Best Params for this fold: {'isomap__n_components': 3, 'isomap__n_neighbors': 10, 'kernel_ridge__alpha': 10, 'kernel_ridge__gamma': 1}
  Calculating metrics on the outer test set...
    R2: 0.6707

Performing Outer Fold 2/5
Performing GridSearchCV (optimizing for 'r2')...
  Best Params for this fold: {'isomap__n_components': 3, 'isomap__n_neighbors': 10, 'kernel_ridge__alpha': 10, 'kernel_ridge__gamma': 0.01}
  Calculating metrics on the outer test set...
    R2: 0.7349

Performing Outer Fold 3/5
Performing GridSearchCV (optimizing for 'r2')...
  Best Params for this fold: {'isomap__n_components': 3, 'isomap__n_neighbors': 10, 'kernel_ridge__alpha': 0.01, 'kernel_ridge__gamma': 0.1}
  Calculating metrics on the outer test set...
    R2: 0.7331

Performing Outer Fold 4/5
Performing GridSearchCV (optimizing for 'r2')...
  Best Params for this fold: {'isomap__n_components': 3, 'i

# Train final model with Kernel Ridge Regressor

In [6]:
final_model_kr, final_params_kr, test_metrics_kr = train_final_model_from_nested_cv(
    model = kr_pipeline,
    all_fold_best_params = kr_result_nested_cv["all_fold_best_params"],
    X = X_train,
    y = y_train,
    X_test = X_test,
    y_test = y_test,
    score_per_fold = kr_result_nested_cv["performance_summary"]["r2"]["all_scores"],
    strategy = 'most_frequent',
    scoring = ['r2']
)


[STRATEGIA: most_frequent] Parametri più frequenti sui fold:
{'kernel_ridge__alpha': 10, 'isomap__n_neighbors': 10, 'isomap__n_components': 3, 'kernel_ridge__gamma': 0.01}

Calcolo delle metriche sul test set finale...
  R2: 0.6992


## Pipeline 2
- Preprocessing: MinMaxScaler
- Riduzione dimensionalità: LocallyLinearEmbedding (n_components=2)
- Modello: DecisionTreeRegressor (ottimizzazione di max_depth)


In [7]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.manifold import LocallyLinearEmbedding
from sklearn.tree import DecisionTreeRegressor

dtr_pipeline = Pipeline([
    ('scaler', MinMaxScaler()),
    ('lle', LocallyLinearEmbedding()),
    ('dtr', DecisionTreeRegressor())
])

dtr_params_grid = {
    'scaler__feature_range': [(-1, 1), (0, 1)],
    'lle__n_neighbors': [3, 5, 10],
    'lle__n_components': [2, 3],
    'dtr__max_depth': [3, 5, 10]
}

In [8]:
dtr_nested_cv_result = nested_cv(
    model = dtr_pipeline,
    param_grid = dtr_params_grid,
    X_train = X_train,
    y_train = y_train,
    outer_splits = 5,
    inner_splits = 3,
    scoring = ['r2']
)

--- Start time: 0.00 seconds ---

Performing Outer Fold 1/5
Performing GridSearchCV (optimizing for 'r2')...
  Best Params for this fold: {'dtr__max_depth': 10, 'lle__n_components': 3, 'lle__n_neighbors': 5, 'scaler__feature_range': (-1, 1)}
  Calculating metrics on the outer test set...
    R2: 0.9232

Performing Outer Fold 2/5
Performing GridSearchCV (optimizing for 'r2')...
  Best Params for this fold: {'dtr__max_depth': 10, 'lle__n_components': 3, 'lle__n_neighbors': 5, 'scaler__feature_range': (0, 1)}
  Calculating metrics on the outer test set...
    R2: 0.3372

Performing Outer Fold 3/5
Performing GridSearchCV (optimizing for 'r2')...
  Best Params for this fold: {'dtr__max_depth': 10, 'lle__n_components': 3, 'lle__n_neighbors': 5, 'scaler__feature_range': (-1, 1)}
  Calculating metrics on the outer test set...
    R2: 0.6511

Performing Outer Fold 4/5
Performing GridSearchCV (optimizing for 'r2')...
  Best Params for this fold: {'dtr__max_depth': 10, 'lle__n_components': 3, 'll

# Train final model with Decision Tree Regressor

In [12]:
final_model_dtr, final_params_dtr, test_metrics_dtr = train_final_model_from_nested_cv(
    model = dtr_pipeline,
    all_fold_best_params = dtr_nested_cv_result["all_fold_best_params"],
    X = X_train,
    y = y_train,
    X_test = X_test,
    y_test = y_test,
    score_per_fold = dtr_nested_cv_result["performance_summary"]["r2"]["all_scores"],
    strategy = 'most_frequent',
    scoring = ['r2']
)


[STRATEGIA: most_frequent] Parametri più frequenti sui fold:
{'dtr__max_depth': 10, 'scaler__feature_range': (-1, 1), 'lle__n_components': 3, 'lle__n_neighbors': 5}

Calcolo delle metriche sul test set finale...
  R2: 0.6164
